In [1]:
import sklearn
print(sklearn.__version__)

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
# Other plots (example)

from src.exp import (
    FeatureSchema, ExperimentConfig, ExperimentFacade,
    DataReadConfig, PlotManager
    
)

In [2]:
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    print("Intel sklearn patch enabled")
except ImportError:
    print("sklearnex not installed; using standard sklearn")


sklearnex not installed; using standard sklearn


In [3]:
schema = FeatureSchema(
    target="price",
    num_cols=["year","engineSize","mpg","tax","mileage"],
    cat_cols=["model","fuelType","transmission"]
)


In [4]:
cfg = ExperimentConfig(outer_folds=5, inner_folds=5, n_trials=40, seed=42, log_target=True)

In [5]:
data_cfg = DataReadConfig(
    root_dir="Dataset/data",
    recursive=True,
    exclude_filenames=["cclass.csv", "unclean focus.csv","unclean cclass.csv","focus.csv"],  # the excluded files
    add_source_column=True,
    encoding="utf-8",
)

In [6]:
models = ["LinearRegression", "DecisionTree", "RandomForest", "SVR", "XGBoost", "NeuralNetwork"]

In [7]:
exp = ExperimentFacade.from_folder(
    data_cfg=data_cfg,
    schema=schema,
    cfg=cfg,
    model_names=models,
    hparam_json="config/hyperparams.json"
)

In [ ]:
results = exp.run()

[I 2026-01-24 19:36:13,459] A new study created in memory with name: LinearRegression_OuterFold_1
[I 2026-01-24 19:36:14,376] Trial 0 finished with value: -2.9745302473682678 and parameters: {}. Best is trial 0 with value: -2.9745302473682678.
[I 2026-01-24 19:36:14,420] Trial 1 finished with value: -2.9745302473682678 and parameters: {}. Best is trial 0 with value: -2.9745302473682678.
[I 2026-01-24 19:36:14,462] Trial 2 finished with value: -2.9745302473682678 and parameters: {}. Best is trial 0 with value: -2.9745302473682678.
[I 2026-01-24 19:36:14,502] Trial 3 finished with value: -2.9745302473682678 and parameters: {}. Best is trial 0 with value: -2.9745302473682678.
[I 2026-01-24 19:36:14,545] Trial 4 finished with value: -2.9745302473682678 and parameters: {}. Best is trial 0 with value: -2.9745302473682678.
[I 2026-01-24 19:36:14,588] Trial 5 finished with value: -2.9745302473682678 and parameters: {}. Best is trial 0 with value: -2.9745302473682678.
[I 2026-01-24 19:36:14,805

In [ ]:
display(exp.summary())


In [ ]:
sig = exp.significance(
    metric="R2",
    baseline="RandomForest",
    models=["RandomForest", "XGBoost", "SVR"]
)

In [ ]:
display(sig)

In [ ]:
shap_analyzer = exp.shap(models=models)

In [ ]:
for m in shap_analyzer.available_models():
    shap_analyzer.beeswarm(m)

In [ ]:
def plot_model_comparison(df_summary, plot_manager=None):

    df_summary.set_index("model")["R2_mean"].plot(kind="bar")
    plt.ylabel("R2")

    if plot_manager is not None:
        plot_manager.save("model_comparison_r2")
    plt.show()


In [ ]:

pm = PlotManager("outputs/figures/metrics")
plot_model_comparison(exp.summary(), plot_manager=pm)

In [ ]:
significance_matrix = exp.significance_matrix(metric="R2")
display(significance_matrix)

In [ ]:
exp.save_best_params()